In [1]:

!pip install "pymilvus[milvus_lite]"

In [3]:
# Install (run once if needed)
# !pip install pymilvus milvus-lite sentence-transformers pandas matplotlib

from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility
from sentence_transformers import SentenceTransformer
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 1. Connect to Milvus Lite
# -----------------------------
connections.connect("default", uri="./milvus_demo.db")

# -----------------------------
# 2. Define / Load Collection
# -----------------------------
collection_name = "demo_collection"

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384)
]

schema = CollectionSchema(fields, description="Text search demo")

# Check if collection exists
if utility.has_collection(collection_name):
    collection = Collection(collection_name)
else:
    collection = Collection(name=collection_name, schema=schema)

# -----------------------------
# 3. Load Embedding Model
# -----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

# -----------------------------
# 4. Sample Data (10 rows)
# -----------------------------
texts = [
    "I love machine learning",
    "Artificial intelligence is amazing",
    "Deep learning is a subset of AI",
    "I enjoy playing football",
    "Cricket is popular in India",
    "Neural networks are powerful",
    "I like watching movies",
    "Data science is interesting",
    "Python is great for AI",
    "Football and cricket are sports"
]

# Convert to embeddings
embeddings = model.encode(texts)

# -----------------------------
# 5. Insert Data (only if empty)
# -----------------------------
if collection.num_entities == 0:
    ids = list(range(len(texts)))
    collection.insert([ids, embeddings.tolist()])
    collection.flush()

# Load collection into memory
collection.load()

# -----------------------------
# 6. Search Query
# -----------------------------
query = "AI and machine learning"
query_embedding = model.encode([query])

results = collection.search(
    data=query_embedding.tolist(),
    anns_field="embedding",
    param={"metric_type": "L2", "params": {"nprobe": 10}},
    limit=3
)

# -----------------------------
# 7. Display Results
# -----------------------------
output = []
for hits in results:
    for hit in hits:
        output.append({
            "id": hit.id,
            "distance": hit.distance,
            "text": texts[hit.id]
        })

df = pd.DataFrame(output)
print("🔍 Search Results:")
display(df)

# -----------------------------
# 8. Visualization
# -----------------------------
plt.bar(range(len(df)), df["distance"])
plt.title("Similarity Distance (Lower = More Similar)")
plt.xlabel("Result Rank")
plt.ylabel("Distance")
plt.show()

ConnectionConfigException: <ConnectionConfigException: (code=1, message=milvus-lite is required for local database connections. Please install it with: pip install pymilvus[milvus_lite])>